In [1]:
import os


def resolve_path(filename: str) -> str:
    """Finds the file whether it's in the current folder or a 'test' subfolder."""
    if os.path.exists(filename):
        return filename
    test_subfolder = os.path.join("test", filename)
    if os.path.exists(test_subfolder):
        return test_subfolder
    # Check parent directory or stripped name if Windows added numbers
    base_name = os.path.basename(filename)
    if os.path.exists(base_name):
        return base_name
    return filename


def load_stopwords(filepath: str) -> set:
    """Reads a stopwords file into a set of lowercase words."""
    path = resolve_path(filepath)
    stopwords = set()
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            word = line.strip().lower()
            if word:
                stopwords.add(word)
    return stopwords


def process_document(filepath: str, stopwords: set) -> list:
    """Tokenizes text, strips punctuation, and filters out stopwords."""
    path = resolve_path(filepath)
    remaining_words = []
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            for word in line.lower().split():
                clean_word = word.strip(".,!?'\":;")
                if clean_word and clean_word not in stopwords:
                    remaining_words.append(clean_word)
    return remaining_words


def compute_term_frequencies(words: list) -> dict:
    """Computes relative term frequencies: count / total_words."""
    if not words:
        return {}
    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1
    total_words = len(words)
    return {word: count / total_words for word, count in counts.items()}


def run_analysis(stopwords_path: str, doc1_path: str, doc2_path: str):
    stopwords = load_stopwords(stopwords_path)
    doc1_words = process_document(doc1_path, stopwords)
    doc2_words = process_document(doc2_path, stopwords)

    doc1_tf = compute_term_frequencies(doc1_words)
    doc2_tf = compute_term_frequencies(doc2_words)

    set1 = set(doc1_tf.keys())
    set2 = set(doc2_tf.keys())

    common_words = set1.intersection(set2)
    unique_doc1 = {w: doc1_tf[w] for w in (set1 - set2)}
    unique_doc2 = {w: doc2_tf[w] for w in (set2 - set1)}

    print("Remaining Words in Document 1:")
    print(doc1_words)
    print("\nRemaining Words in Document 2:")
    print(doc2_words)
    print("\nTerm Frequencies for Document 1:")
    print(doc1_tf)
    print("\nTerm Frequencies for Document 2:")
    print(doc2_tf)
    print("\nCommon words:")
    print(common_words)
    print("\nUnique words in Document 1:")
    print(unique_doc1)
    print("\nUnique words in Document 2:")
    print(unique_doc2)

    return {
        "doc1_words": doc1_words,
        "doc2_words": doc2_words,
        "doc1_tf": doc1_tf,
        "doc2_tf": doc2_tf,
        "common": common_words,
        "unique1": unique_doc1,
        "unique2": unique_doc2,
    }

In [2]:
# Step 1 & 2 Verification: Built-in Verification Test
test_stopwords_data = "a\nthe\nis\nin\nand\n"
test_doc1_data = "The small cat and big dog."
test_doc2_data = "A dog is fast, and a cat is slow."

with open("test_stopwords.txt", "w", encoding="utf-8") as f:
    f.write(test_stopwords_data)
with open("test_doc1.txt", "w", encoding="utf-8") as f:
    f.write(test_doc1_data)
with open("test_doc2.txt", "w", encoding="utf-8") as f:
    f.write(test_doc2_data)

print("=== Running Verification Test Suite ===")
t1_results = run_analysis("test_stopwords.txt", "test_doc1.txt", "test_doc2.txt")

# Test validation
assert t1_results["common"] == {"cat", "dog"}, "Common words verification failed!"
print("\n[PASS] Test logic and outputs match expected specification.")

=== Running Verification Test Suite ===
Remaining Words in Document 1:
['small', 'cat', 'big', 'dog']

Remaining Words in Document 2:
['dog', 'fast', 'cat', 'slow']

Term Frequencies for Document 1:
{'small': 0.25, 'cat': 0.25, 'big': 0.25, 'dog': 0.25}

Term Frequencies for Document 2:
{'dog': 0.25, 'fast': 0.25, 'cat': 0.25, 'slow': 0.25}

Common words:
{'cat', 'dog'}

Unique words in Document 1:
{'big': 0.25, 'small': 0.25}

Unique words in Document 2:
{'slow': 0.25, 'fast': 0.25}

[PASS] Test logic and outputs match expected specification.
